# Laboratorium 3: Transfer learning w klasyfikacji obrazów

Klasyfikujemy 37 ras psów i kotów ze zbioru Oxford-IIIT Pet (3680 zdjęć treningowych, 3669 testowych). Najpierw mała sieć konwolucyjna uczona od zera, potem MobileNetV2 wytrenowana na ImageNet jako zamrożony ekstraktor cech, na końcu fine-tuning górnych warstw. Model zapisany w zadaniu 5 będzie potrzebny w laboratorium 4 (Grad-CAM).

Środowisko: Google Colab z GPU. Przed uruchomieniem wybierzcie Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU. Wszystkie potrzebne biblioteki są w Colabie, niczego nie instalujemy.

In [ ]:
import gc
import io
import os
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from PIL import Image
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.dpi"] = 80

print("Keras", keras.__version__, "| backend", keras.backend.backend(), "| TensorFlow", tf.__version__)
gpu = tf.config.list_physical_devices("GPU")
print("GPU:", gpu if gpu else "brak, obliczenia na CPU (trening będzie wolny)")

In [ ]:
# Liczby epok dla Colaba z GPU (jak w instrukcji)
EPOKI_BASELINE = 15
EPOKI_GLOWA = 10
EPOKI_FT = 10
EPOKI_POROWNANIE = 5   # zadania rozszerzone
EPOKI_MINI = 15        # zadania rozszerzone
BATCH = 32
ROZMIAR = 160

## Zadanie 1: Dane

Komórka poniżej jest gotowa. Pobiera dwa pliki parquet z Hugging Face (razem około 790 MB, kilkanaście do kilkudziesięciu sekund), dekoduje zdjęcia JPEG po kolei, konwertuje je do RGB i skaluje do 160x160 pikseli. Wynikiem są tablice `uint8`. Z części treningowej wydzielamy 20% na walidację ze stratyfikacją, część testowa zostaje osobno i użyjemy jej dopiero w zadaniu 4.

1. Policzcie zdjęcia w każdej klasie w train, walidacji i teście. Narysujcie rozkład klas w train.
2. Pokażcie siatkę 12 losowych zdjęć z nazwami ras.
3. Odpowiedzcie na pytania pod zadaniem.

In [ ]:
# Nazwy klas w kolejności etykiet 0..36 (README zbioru timm/oxford-iiit-pet na Hugging Face).
# 12 ras kotów: abyssinian, bengal, birman, bombay, british_shorthair, egyptian_mau, maine_coon,
# persian, ragdoll, russian_blue, siamese, sphynx; pozostałe 25 to rasy psów.
KLASY = [
    "abyssinian", "american_bulldog", "american_pit_bull_terrier", "basset_hound", "beagle",
    "bengal", "birman", "bombay", "boxer", "british_shorthair", "chihuahua", "egyptian_mau",
    "english_cocker_spaniel", "english_setter", "german_shorthaired", "great_pyrenees", "havanese",
    "japanese_chin", "keeshond", "leonberger", "maine_coon", "miniature_pinscher", "newfoundland",
    "persian", "pomeranian", "pug", "ragdoll", "russian_blue", "saint_bernard", "samoyed",
    "scottish_terrier", "shiba_inu", "siamese", "sphynx", "staffordshire_bull_terrier",
    "wheaten_terrier", "yorkshire_terrier",
]
URL = "https://huggingface.co/datasets/timm/oxford-iiit-pet/resolve/main/data/{}-00000-of-00001.parquet"


def wczytaj_czesc(nazwa):
    """Pobiera część zbioru (train albo test) i zwraca X (N, 160, 160, 3) uint8 oraz etykiety y."""
    df = pd.read_parquet(URL.format(nazwa), columns=["image", "label"])
    X = np.empty((len(df), ROZMIAR, ROZMIAR, 3), dtype=np.uint8)
    for i, obraz in enumerate(df["image"]):
        with Image.open(io.BytesIO(obraz["bytes"])) as img:
            # zwykłe skalowanie do kwadratu, proporcje zdjęcia nie są zachowane
            X[i] = np.asarray(img.convert("RGB").resize((ROZMIAR, ROZMIAR), Image.BILINEAR))
    y = df["label"].to_numpy().astype("int64")
    del df
    gc.collect()
    return X, y


t0 = time.perf_counter()
X_trainval, y_trainval = wczytaj_czesc("train")
X_test, y_test = wczytaj_czesc("test")
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, stratify=y_trainval, random_state=RANDOM_STATE)
del X_trainval, y_trainval
gc.collect()
print(f"Wczytano w {time.perf_counter() - t0:.0f} s")
print("train", X_train.shape, "| walidacja", X_val.shape, "| test", X_test.shape, "| typ", X_train.dtype)

In [ ]:
# Funkcje pomocnicze (gotowe)

def pokaz_obrazy(obrazy, podpisy, kolumny=6, tytul=None):
    wiersze = int(np.ceil(len(obrazy) / kolumny))
    fig, osie = plt.subplots(wiersze, kolumny, figsize=(1.6 * kolumny, 1.9 * wiersze))
    for i, ax in enumerate(np.ravel(osie)):
        ax.axis("off")
        if i < len(obrazy):
            ax.imshow(obrazy[i])
            ax.set_title(podpisy[i], fontsize=7)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def rysuj_krzywe(historia, tytul, granica=None):
    """historia: słownik z listami accuracy, val_accuracy, loss, val_loss (np. model.fit(...).history)."""
    epoki = np.arange(1, len(historia["loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, klucz, nazwa in [(ax1, "accuracy", "Dokładność"), (ax2, "loss", "Funkcja straty")]:
        ax.plot(epoki, historia[klucz], "o-", label="trening")
        ax.plot(epoki, historia["val_" + klucz], "o-", label="walidacja")
        if granica is not None:
            ax.axvline(granica + 0.5, color="gray", linestyle="--", label="początek fine-tuningu")
        ax.set_xlabel("Epoka")
        ax.set_ylabel(nazwa)
        ax.set_title(nazwa)
        ax.legend()
    fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


class CzasEpoki(keras.callbacks.Callback):
    """Zapisuje czas każdej epoki w sekundach (razem z walidacją) w liście self.czasy."""

    def on_train_begin(self, logs=None):
        self.czasy = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.czasy.append(time.perf_counter() - self._start)


def nowa_augmentacja():
    """Nowy blok augmentacji; działa tylko w trakcie treningu, przy predykcji przepuszcza obraz bez zmian."""
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name="augmentacja")

In [ ]:
# TODO 1.1: policzcie zdjęcia w każdej klasie (np.bincount) w train, walidacji i teście,
# wypiszcie minimum i maksimum, a dla train narysujcie wykres słupkowy z nazwami ras na osi X.

In [ ]:
# TODO 1.2: wylosujcie 12 zdjęć z X_train (np.random.default_rng(RANDOM_STATE)) i pokażcie je
# funkcją pokaz_obrazy z nazwami ras jako podpisami.

### Pytania do zadania 1

1. Ile zdjęć przypada średnio na jedną rasę w zbiorze treningowym? Czy zbiór jest zbalansowany?
2. Dlaczego trening od zera na około 3 tys. zdjęć i 37 klasach jest trudny?
3. Które rasy na siatce lub z waszej wiedzy łatwo pomylić?
4. Co tracimy, skalując każde zdjęcie do kwadratu 160x160 bez zachowania proporcji?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 2: Baseline, mała CNN od zera

1. Zbudujcie małą sieć konwolucyjną z augmentacją (`nowa_augmentacja()`), przeskalowaniem `Rescaling(1/255)`, kilkoma blokami `Conv2D` + `MaxPooling2D`, `GlobalAveragePooling2D`, `Dropout` i `Dense(37, softmax)`.
2. Wytrenujcie ją przez `EPOKI_BASELINE` epok i zmierzcie dokładność walidacyjną. Nie zgadujcie wyniku, zmierzcie go.
3. Narysujcie krzywe uczenia.

In [ ]:
# TODO 2.1: zbudujcie małą CNN (API funkcyjne): wejście 160x160x3, nowa_augmentacja(), Rescaling(1/255),
# 3 do 4 bloków Conv2D (relu, padding="same") + MaxPooling2D, GlobalAveragePooling2D, Dropout, Dense(37, softmax).
# Skompilujcie z Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"]. Wypiszcie liczbę parametrów.
model_cnn = None

In [ ]:
czas_cnn = CzasEpoki()
# TODO 2.2: wytrenujcie model_cnn: model_cnn.fit(X_train, y_train, validation_data=(X_val, y_val),
# epochs=EPOKI_BASELINE, batch_size=BATCH, callbacks=[czas_cnn], verbose=2). Zapiszcie najlepszą
# dokładność walidacyjną w acc_cnn, wypiszcie średni czas epoki i narysujcie krzywe (rysuj_krzywe).
historia_cnn = None
acc_cnn = None

### Pytania do zadania 2

1. Jaką dokładność walidacyjną osiągnęła sieć i ile to razy więcej niż zgadywanie (1/37)?
2. Czy model się przeucza, czy raczej jest niedouczony? Na co patrzycie na wykresach?
3. Co mogłoby poprawić wynik sieci uczonej od zera i ile by to kosztowało (czas, dane)?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 3: MobileNetV2 jako ekstraktor cech

1. Wczytajcie `keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(160, 160, 3))` i zamroźcie ją (`baza.trainable = False`). Wagi (około 9 MB) pobierają się przy pierwszym wywołaniu.
2. Zbudujcie model: augmentacja, przeskalowanie wejścia do zakresu [-1, 1] warstwą `Rescaling(1/127.5, offset=-1)` (to samo robi `keras.applications.mobilenet_v2.preprocess_input`), wywołanie `baza(x, training=False)`, `GlobalAveragePooling2D`, `Dropout(0.2)`, `Dense(37, softmax)`.
3. Wytrenujcie samą głowę przez `EPOKI_GLOWA` epok. Callback `checkpoint` zapisuje wagi z najlepszą dokładnością walidacyjną; ten sam obiekt użyjemy w zadaniu 4, więc zapamięta najlepszy wynik z obu etapów.
4. Porównajcie wynik z baseline.

Dlaczego `training=False`? Baza zawiera warstwy BatchNormalization. W trybie treningowym normalizują one dane statystykami bieżącego batcha i aktualizują średnie kroczące zapamiętane z ImageNet. Przewodnik Keras zaleca wywołanie `baza(x, training=False)`, żeby utrzymać je w trybie wnioskowania także po odmrożeniu warstw. Uwaga: w Keras 3.13.2, na którym przygotowano rozwiązanie, `fit` przekazuje `training=True` do zagnieżdżonej bazy i nadpisuje ten argument (sprawdzone na małym przykładzie: średnie kroczące odmrożonej warstwy BatchNormalization zmieniały się mimo `training=False`). Dopóki baza jest zamrożona, nie ma to znaczenia, bo warstwa BatchNormalization z `trainable=False` zawsze działa w trybie wnioskowania. Dlatego w zadaniu 4 warstwy BatchNormalization zostawimy zamrożone jawnie, a `training=False` zostaje jako zapis intencji.

In [ ]:
# TODO 3.1: wczytajcie bazę MobileNetV2 (include_top=False, weights="imagenet",
# input_shape=(ROZMIAR, ROZMIAR, 3)) i zamroźcie ją. Wypiszcie liczbę warstw i parametrów bazy.
baza = None

In [ ]:
# TODO 3.2: zbudujcie model (API funkcyjne): wejście, nowa_augmentacja(), Rescaling(1/127.5, offset=-1),
# baza(x, training=False), GlobalAveragePooling2D, Dropout(0.2), Dense(37, softmax).
# Skompilujcie z Adam(1e-3). Wypiszcie liczbę wszystkich parametrów i parametrów trenowanych
# (sum(int(np.prod(w.shape)) for w in model.trainable_weights)).
model = None

In [ ]:
PLIK_WAG = "lab03_najlepsze.weights.h5"
checkpoint = keras.callbacks.ModelCheckpoint(PLIK_WAG, monitor="val_accuracy",
                                             save_best_only=True, save_weights_only=True)
czas_glowa = CzasEpoki()
# TODO 3.3: wytrenujcie głowę przez EPOKI_GLOWA epok z callbacks=[checkpoint, czas_glowa] (reszta jak w 2.2).
# Zapiszcie najlepszą dokładność walidacyjną w acc_glowa, narysujcie krzywe i zestawcie wynik z acc_cnn w tabeli.
historia_glowa = None
acc_glowa = None

### Pytania do zadania 3

1. Ile parametrów trenujecie w tym zadaniu, a ile w CNN od zera? Skąd tak duża różnica w wyniku?
2. Po co `training=False` przy wywołaniu bazy i dlaczego w zadaniu 4 zamrażamy dodatkowo warstwy BatchNormalization? Co by się stało z ich statystykami, gdyby po odmrożeniu działały w trybie treningowym?
3. Dlaczego wejście trzeba przeskalować dokładnie tak, jak przy treningu bazy na ImageNet? Co by się stało przy `Rescaling(1/255)`?
4. Dlaczego dokładność walidacyjna bywa tu wyższa niż treningowa?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 4: Fine-tuning

1. Odmroźcie bazę (`baza.trainable = True`), a potem ponownie zamroźcie warstwy `baza.layers[:FINE_TUNE_OD]` oraz wszystkie warstwy BatchNormalization (patrz uwaga w zadaniu 3). Przy `FINE_TUNE_OD = 110` odmrażamy 44 ostatnie warstwy ze 154, z wyjątkiem BatchNormalization.
2. Skompilujcie model ponownie z niskim learning rate (`Adam(1e-5)`). Bez ponownej kompilacji zmiana `trainable` nie zadziała.
3. Trenujcie dalej przez `EPOKI_FT` epok (`initial_epoch=EPOKI_GLOWA`) z tym samym obiektem `checkpoint`. Narysujcie krzywe obu etapów na jednym wykresie.
4. Wczytajcie najlepsze wagi i raz policzcie dokładność na zbiorze testowym.

In [ ]:
FINE_TUNE_OD = 110   # indeks pierwszej odmrażanej warstwy bazy (MobileNetV2 ma 154 warstwy)
# TODO 4.1: odmroźcie bazę (baza.trainable = True), zamroźcie z powrotem baza.layers[:FINE_TUNE_OD]
# i wszystkie warstwy BatchNormalization (isinstance(warstwa, layers.BatchNormalization)),
# skompilujcie model ponownie z Adam(1e-5). Ile warstw i parametrów jest teraz trenowanych?
# Zapamiętajcie moving_mean ostatniej BatchNormalization, a po treningu sprawdźcie, że się nie zmieniła.

In [ ]:
czas_ft = CzasEpoki()
# TODO 4.2: trenujcie dalej: epochs=EPOKI_GLOWA + EPOKI_FT, initial_epoch=EPOKI_GLOWA, callbacks=[checkpoint, czas_ft].
# Połączcie historie obu etapów: {k: historia_glowa.history[k] + historia_ft.history[k] for k in [...]}
# i narysujcie je funkcją rysuj_krzywe(..., granica=EPOKI_GLOWA). Zapiszcie najlepszą val_accuracy z tego etapu w acc_ft.
historia_ft = None
acc_ft = None

In [ ]:
# TODO 4.3: wczytajcie najlepsze wagi (model.load_weights(PLIK_WAG)), raz policzcie predykcje na X_test
# (model.predict, batch_size=64), y_pred = argmax i dokładność testową acc_test. Zestawcie w tabeli
# najlepszą val_accuracy: CNN od zera, ekstraktor cech, fine-tuning.
y_pred = None

### Pytania do zadania 4

1. Dlaczego fine-tuning wymaga niskiego learning rate? Co by się stało przy 1e-3?
2. Dlaczego najpierw trenujemy samą głowę, a dopiero potem odmrażamy część bazy?
3. O ile fine-tuning poprawił dokładność walidacyjną? Czy przy tej liczbie epok było warto?
4. Dlaczego zbiór testowy liczymy raz, dla jednego wybranego modelu, a nie po każdym eksperymencie?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 5: Analiza błędów i zapis modelu

1. Policzcie macierz pomyłek na zbiorze testowym (predykcje z zadania 4) i wypiszcie 5 par ras mylonych najczęściej (pomyłki w obie strony razem).
2. Pokażcie 12 błędnie sklasyfikowanych zdjęć z prawdziwą i przewidzianą rasą.
3. Zapiszcie model: `model.save("lab03_pets.keras")`. Plik będzie potrzebny w laboratorium 4. W Colabie pobierzcie go na dysk (panel Pliki, Pobierz, albo `from google.colab import files; files.download("lab03_pets.keras")`) lub skopiujcie na Google Drive, bo po zamknięciu sesji zniknie.

In [ ]:
# TODO 5.1: macierz pomyłek: cm = confusion_matrix(y_test, y_pred, labels=range(len(KLASY))).
# Narysujcie ją (plt.imshow, nazwy ras na osiach) i wypiszcie 5 par ras mylonych najczęściej:
# dla i < j zsumujcie cm[i, j] + cm[j, i] i posortujcie malejąco.

In [ ]:
# TODO 5.2: wybierzcie 12 losowych indeksów, dla których y_pred != y_test, i pokażcie zdjęcia
# funkcją pokaz_obrazy z podpisami "prawda: ...\nmodel: ...".

In [ ]:
# TODO 5.3: zapiszcie model (model.save("lab03_pets.keras")), wczytajcie go keras.models.load_model
# i sprawdźcie, że dla 16 pierwszych zdjęć testowych daje te same predykcje co p_test[:16] (np.allclose).
# W Colabie pobierzcie plik albo skopiujcie go na Google Drive.

### Pytania do zadania 5

1. Czy najczęściej mylone pary mają sens z punktu widzenia wyglądu ras? Czy model myli psa z kotem?
2. Czego nie widać w samej dokładności, a widać w macierzy pomyłek?
3. Co zrobilibyście dalej, żeby poprawić model (dane, rozdzielczość, architektura, trening)?

Odpowiedzi:

1. 
2. 
3. 
4. 

# Zadania rozszerzone

Dwa zadania wykraczające poza podstawowy zakres laboratorium.

## Zadanie 6: Porównanie baz jako ekstraktorów cech

1. Napiszcie funkcję `zbuduj_ekstraktor(baza, przetworzenie, liczba_klas)` z tą samą głową co w zadaniu 3.
2. Wytrenujcie trzy modele z zamrożonymi bazami MobileNetV2, EfficientNetB0 i ResNet50, z tym samym budżetem (`EPOKI_POROWNANIE` epok, ten sam batch, te same dane).
3. Uwaga na przetwarzanie wejścia: MobileNetV2 oczekuje [-1, 1], EfficientNetB0 w Keras ma skalowanie i normalizację wbudowane w model i oczekuje pikseli 0..255, ResNet50 ma własne `keras.applications.resnet50.preprocess_input` (zamiana RGB na BGR i odjęcie średnich z ImageNet).
4. Zestawcie w tabeli: najlepszą dokładność walidacyjną, liczbę parametrów bazy, liczbę parametrów trenowanych i średni czas epoki.

In [ ]:
# TODO 6.1: funkcja zbuduj_ekstraktor(baza, przetworzenie, liczba_klas): baza.trainable = False, wejście,
# nowa_augmentacja(), przetworzenie(x) (jeśli nie None), baza(x, training=False), GAP, Dropout(0.2),
# Dense(liczba_klas, softmax), kompilacja z Adam(1e-3). Zwraca skompilowany model.

In [ ]:
# TODO 6.2: dla każdej bazy (MobileNetV2, EfficientNetB0, ResNet50) ustawcie keras.utils.set_random_seed(RANDOM_STATE),
# wczytajcie bazę z weights="imagenet", zbudujcie model, trenujcie EPOKI_POROWNANIE epok z CzasEpoki.
# Zbierzcie w DataFrame: najlepszą val_accuracy, liczbę parametrów bazy, parametry trenowane, średni czas epoki.
wyniki_baz = None

### Pytania do zadania 6

1. Która baza dała najlepszy wynik przy tym samym budżecie? Czy wynik idzie w parze z liczbą parametrów?
2. Ile kosztuje każda baza w czasie epoki? Którą wybralibyście do aplikacji na telefonie, a którą do serwera?
3. Co by się stało, gdyby EfficientNetB0 dostała wejście przeskalowane do [-1, 1], a ResNet50 piksele 0..255 bez `preprocess_input`?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 7: Własny mały zbiór

Zróbcie telefonem lub zbierzcie po około 30 zdjęć dla 3 kategorii (na przykład trzy własne przedmioty). Ułóżcie je w katalogach `moje_zdjecia/kategoria_a/*.jpg`, `moje_zdjecia/kategoria_b/*.jpg`, `moje_zdjecia/kategoria_c/*.jpg`, spakujcie do `moje_zdjecia.zip`, wgrajcie do Colaba (panel Pliki) i rozpakujcie poleceniem `!unzip -q moje_zdjecia.zip`.

1. Wczytajcie zdjęcia funkcją `keras.utils.image_dataset_from_directory` z podziałem 80/20 na trening i walidację.
2. Zbudujcie ekstraktor cech MobileNetV2 dla 3 klas (funkcja z zadania 6) i wytrenujcie go przez `EPOKI_MINI` epok.
3. Oceńcie wynik na walidacji i pokażcie kilka zdjęć z predykcjami.

Kto nie ma własnych zdjęć, może użyć gotowej funkcji `przygotuj_zbior_zastepczy`, która zapisuje po 30 zdjęć z 3 ras Oxford Pets w tej samej strukturze katalogów.

In [ ]:
import shutil
import tempfile


def przygotuj_zbior_zastepczy(klasy, n_na_klase=30, katalog=None):
    """Zapisuje po n_na_klase zdjęć z części train Oxford Pets jako katalog/klasa/NNN.jpg i zwraca ścieżkę katalogu."""
    katalog = katalog or tempfile.mkdtemp(prefix="lab03_mini_")
    df = pd.read_parquet(URL.format("train"), columns=["image", "label"])
    for nazwa in klasy:
        os.makedirs(os.path.join(katalog, nazwa), exist_ok=True)
        wiersze = df[df["label"] == KLASY.index(nazwa)].sample(n_na_klase, random_state=RANDOM_STATE)
        for i, obraz in enumerate(wiersze["image"]):
            with Image.open(io.BytesIO(obraz["bytes"])) as img:
                img.convert("RGB").save(os.path.join(katalog, nazwa, f"{i:03d}.jpg"), quality=90)
    del df
    gc.collect()
    return katalog

In [ ]:
KATALOG_MINI = "moje_zdjecia"   # struktura: moje_zdjecia/kategoria/*.jpg
if not os.path.isdir(KATALOG_MINI):
    print(f"Brak katalogu {KATALOG_MINI}. Bez własnych zdjęć odkomentujcie linię poniżej (zbiór zastępczy).")
    # KATALOG_MINI = przygotuj_zbior_zastepczy(["beagle", "persian", "samoyed"])

In [ ]:
# TODO 7.1: train_mini, val_mini = keras.utils.image_dataset_from_directory(KATALOG_MINI, validation_split=0.2,
# subset="both", seed=RANDOM_STATE, image_size=(ROZMIAR, ROZMIAR), batch_size=16). Wypiszcie class_names.

In [ ]:
# TODO 7.2: zbudujcie ekstraktor cech MobileNetV2 dla len(train_mini.class_names) klas (zbuduj_ekstraktor
# z zadania 6), trenujcie EPOKI_MINI epok (fit(train_mini, validation_data=val_mini, ...)), oceńcie na walidacji,
# narysujcie krzywe i pokażcie kilka zdjęć walidacyjnych z predykcją.

### Pytania do zadania 7

1. Jaką dokładność osiągnął model na tak małym zbiorze? Jak bardzo możecie ufać wynikowi z kilkunastu zdjęć walidacyjnych?
2. Dlaczego transfer learning działa przy 30 zdjęciach na klasę, a CNN od zera z zadania 2 nie działała przy około 80?
3. Jakie cechy waszych zdjęć (tło, oświetlenie, ten sam stół) mogą sprawić, że model uczy się czegoś innego, niż chcecie?

Odpowiedzi:

1. 
2. 
3. 
4. 